In [1]:
import json
import os
import re
import urllib.request

sample_notes = """Website revamp kickoff

Alex: What are the main features you want on the new landing page?
Client: We need a modern hero section, customer testimonials, and a contact form.
Alex: Would you prefer Calendly integration or a custom-built solution?
Client: Calendly works fine and keeps it simple.
Alex: Perfect, I'll prepare the initial wireframes by Wednesday.
"""

def offline_extract(notes):
    """Extract lines in the format Name: task by deadline."""

    actions = []

    pattern = r"(\w+):\s*(.*?)\s+by\s+([A-Za-z]+)"

    for line in notes.splitlines():

        match = re.search(pattern, line, re.IGNORECASE)

        if match:
            name = match.group(1)
            task = match.group(2)
            deadline = match.group(3)

            actions.append({
                "name": name,
                "task": task,
                "deadline": deadline
            })

    return actions


# Extract action items
result = offline_extract(sample_notes)

# Convert into JSON format
json_output = json.dumps(result, indent=4)

print("Extracted Action Items:")
print(json_output)

# Save results into a JSON file
with open("action_items.json", "w") as file:
    json.dump(result, file, indent=4)

print("\nData saved successfully!")

Extracted Action Items:
[
    {
        "name": "Alex",
        "task": "Perfect, I'll prepare the initial wireframes",
        "deadline": "Wednesday"
    }
]

Data saved successfully!


In [2]:
%pip install ipywidgets

In [3]:
import ipywidgets as widgets

notes_box = widgets.Textarea(
    value=sample_notes,
    description="Notes:",
    layout=widgets.Layout(width="95%", height="170px"),
    style={"description_width": "initial"},
)

mode_box = widgets.ToggleButtons(
    options=[("Offline", "offline"), ("GEMINI FLASH 3.0", "live")],
    value="offline",
    description="Mode:",
)

button = widgets.Button(
    description="Extract action items", 
    button_style="primary", 
    icon="check"
)

output = widgets.Output()

def on_extract_clicked(_):
    with output:
        output.clear_output()
        text = notes_box.value.strip()
        if not text:
            print("Please enter meeting notes first.")
            return

        print("Mode:", "OFFLINE" if mode_box.value == "offline" else "LIVE GEMINI API")
        try:
            actions = offline_extract(text) if mode_box.value == "offline" else live_extract(text)
        except Exception as exc:
            # Keep raw provider errors and credentials off the shared screen
            print(f"Could not extract items ({type(exc).__name__}). Check setup and try again.")
            return

        if not actions:
            print("No actions found. In offline mode, use: Name: task by deadline")
            return

        for i, action in enumerate(actions, 1):
            print(f"\n{i}. {action['task']}")
            print(f"   Owner: {action['owner']} | Due: {action['deadline']}")

        print("\nReview every item against the original notes before using it.")

button.on_click(on_extract_clicked)
display(widgets.VBox([notes_box, mode_box, button, output]))